In [1]:
import pandas as pd

profiles = pd.read_csv("../data/author_profiles.csv")
users = pd.read_csv("../data/preprocessed/main/users.csv")

profiles["author"] = profiles["author"].str.strip().str.lower()

scale_fields = ["author", "test_name", "test_scale", "test_result_type"]
metadata = profiles[scale_fields].drop_duplicates()

scores = users.merge(metadata, on="author", how="left")

print("Users in the main dataset:", users["author"].nunique())

Users in the main dataset: 1311


In [2]:
scale_counts = (
    scores.groupby(["test_name", "test_scale", "test_result_type"], dropna=False)
    ["author"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="users")
)

display(scale_counts.head(20))
rows_per_user = scores["author"].value_counts()
print("Users with multiple metadata rows:", (rows_per_user > 1).sum())

,test_name,test_scale,test_result_type,users
0,truity,percentiles,scores,544
1,understand myself,percentiles,percentiles,273
2,ipip120,percentiles,percentiles,153
3,ipip300,percentiles,percentiles,67
4,personality assessor,percentiles,percentiles,66
5,hexaco,percentiles,scores,35
6,outofservice,percentiles,percentiles,30
7,ipip120,percentiles,description,30
8,understand myself,percentiles,description,28
9,truity,percentiles,description,27


Users with multiple metadata rows: 0


In [3]:
traits = [
    "agreeableness", "openness", "conscientiousness",
    "extraversion", "neuroticism",
]

keep = (
    scores["test_name"].eq("understand myself")
    & scores["test_scale"].eq("percentiles")
    & scores["test_result_type"].eq("percentiles")
)

cohort = scores.loc[keep, ["author"] + traits].copy()
cohort = cohort.sort_values("author").reset_index(drop=True)

print("Users:", len(cohort))
display(cohort[traits].describe().T)
print("Scores outside 0–100:")
print(((cohort[traits] < 0) | (cohort[traits] > 100)).sum())

Users: 273


,count,mean,std,min,25%,50%,75%,max
agreeableness,273.0,37.056777,29.233050,0.0,11.0,32.0,61.0,98.0
openness,273.0,71.990842,24.674814,0.0,57.0,80.0,92.0,99.0
conscientiousness,273.0,38.038462,31.000643,0.0,10.0,31.0,65.0,99.0
extraversion,273.0,48.153846,32.535814,0.0,16.0,50.0,77.0,99.0
neuroticism,273.0,51.040293,33.510208,0.0,18.0,51.0,84.0,99.0


Scores outside 0–100:
agreeableness        0
openness             0
conscientiousness    0
extraversion         0
neuroticism          0
dtype: int64
